This Python code builds a predictive model for podcast listening time using the CatBoost algorithm. It starts by loading training and test data with pandas and preprocesses it by removing outliers in the target variable and filling missing values with the median. Several new features are engineered to potentially improve prediction accuracy, and categorical features are converted to numerical representations using label encoding. The features and target variable are then separated. A CatBoostRegressor model is initialized with specific hyperparameters and trained on the entire training dataset. Finally, the trained model is used to predict listening times for the test data, any negative predictions are clipped to zero, and the results are saved into a submission file in CSV format.

<font color = "DeepSkyBlue">**Import Libraries**

This code imports three essential Python libraries for data science tasks: pandas for data manipulation and analysis (especially working with DataFrames), numpy for numerical computations (handling arrays and mathematical operations), and catboost which provides the CatBoostRegressor class, a gradient boosting algorithm particularly effective with categorical data for regression tasks.

In [ ]:
import pandas as pd
import numpy as np
from catboost import CatBoostRegressor

<font color = "DeepSkyBlue">**Load data**

This section loads the training and test datasets from the specified Kaggle competition input paths using pandas.read_csv(). The training data contains both features and the target variable (Listening_Time_minutes), while the test data includes only the input features to be used for making predictions.

In [ ]:
# Load data
train = pd.read_csv("/kaggle/input/playground-series-s5e4/train.csv")
test = pd.read_csv("/kaggle/input/playground-series-s5e4/test.csv")

<font color = "DeepSkyBlue">**Remove outliers in target**

This line filters out extreme values from the training data by removing any samples where the target variable Listening_Time_minutes is 120 or higher. This helps prevent the model from being influenced by outliers that could skew predictions.

In [ ]:
# Remove outliers in target
train = train[train['Listening_Time_minutes'] < 120]

<font color = "DeepSkyBlue">**Fill missing values**

This part of the code fills any missing values in both the training and test datasets. It uses the median of each numeric column to ensure that the data remains consistent and suitable for training the regression model.

In [ ]:
# Fill missing values
train.fillna(train.median(numeric_only=True), inplace=True)
test.fillna(test.median(numeric_only=True), inplace=True)

<font color = "DeepSkyBlue">**Feature Engineering**

This block creates multiple new features to enhance model performance. It includes differences and weighted combinations of host and guest popularity, as well as the ratio between them. The density and interaction of ads with sentiment and popularity are calculated. Episode length is categorized, and time-based indicators like weekend release and prime-time publication are added. The episode number is also extracted from the title to capture possible progression trends across episodes. These engineered features aim to capture complex relationships that can improve the model’s ability to predict listening time accurately.

In [ ]:
# Feature Engineering (a few new features and interaction)
for df in [train, test]:
    df['Popularity_Diff'] = abs(df['Host_Popularity_percentage'] - df['Guest_Popularity_percentage'])
    df['Combined_Popularity'] = df['Host_Popularity_percentage'] * 0.7 + df['Guest_Popularity_percentage'] * 0.3
    df['Ad_Density'] = df['Number_of_Ads'] / (df['Episode_Length_minutes'] + 0.1)
    df['Length_Category'] = pd.cut(df['Episode_Length_minutes'], bins=[0, 30, 60, 90, 120, 999],
                                   labels=['short', 'medium', 'long', 'vlong', 'extreme']).astype('category').cat.codes
    df['Prime_Time'] = df['Publication_Time'].isin(['Evening', 'Night']).astype(int)
    df['Weekend'] = df['Publication_Day'].isin(['Saturday', 'Sunday']).astype(int)
    df['Sentiment_Score'] = df['Episode_Sentiment'].map({'Positive': 1, 'Neutral': 0, 'Negative': -1})
    df['Ad_Sentiment_Impact'] = df['Number_of_Ads'] * df['Sentiment_Score']
    df['Episode_Seq'] = df['Episode_Title'].str.extract(r'(\d+)').astype(float)
    df['Host_Guest_Ratio'] = df['Host_Popularity_percentage'] / (df['Guest_Popularity_percentage'] + 0.1)
    df['Host_Length_Interaction'] = df['Host_Popularity_percentage'] * df['Episode_Length_minutes']
    df['Guest_Ad_Impact'] = df['Guest_Popularity_percentage'] * df['Number_of_Ads']
    # New features
    df['Popularity_Product'] = df['Host_Popularity_percentage'] * df['Guest_Popularity_percentage']
    df['Length_Sentiment_Interaction'] = df['Episode_Length_minutes'] * df['Sentiment_Score']


<font color = "DeepSkyBlue">**Label Encoding**

This section encodes categorical features into numerical values using label encoding. It converts columns like podcast name, genre, publication day and time, and episode sentiment into integer codes so they can be used effectively by the CatBoost model, which handles numerical inputs efficiently.

In [ ]:
# Label Encoding
cat_cols = ['Podcast_Name', 'Genre', 'Publication_Day', 'Publication_Time', 'Episode_Sentiment']
for col in cat_cols:
    train[col] = train[col].astype('category').cat.codes
    test[col] = test[col].astype('category').cat.codes

<font color = "DeepSkyBlue">**Features and target**

This part separates the features and target variable for model training.
It drops unnecessary columns like id, Listening_Time_minutes, and Episode_Title from the training data to create the input feature matrix X.
The target variable y is extracted as Listening_Time_minutes.
For the test data, id and Episode_Title are also removed to form the final test input X_test for prediction.

In [ ]:
# Features and target
X = train.drop(columns=['id', 'Listening_Time_minutes', 'Episode_Title'])
y = train['Listening_Time_minutes']
X_test = test.drop(columns=['id', 'Episode_Title'])


<font color = "DeepSkyBlue">**Train model directly with slightly adjusted parameters**

This code initializes a CatBoostRegressor model with specific hyperparameters. It sets the number of boosting iterations to 2100 (a slight increase), the learning_rate to 0.019 (a small adjustment), the tree depth to 7 (slightly decreased), the L2 regularization coefficient l2_leaf_reg to 3.5 (a small adjustment), the bagging_temperature for controlling randomness during bagging to 0.45 (a small adjustment), and the random_strength for controlling randomness in tree splits to 0.25 (a small adjustment). The loss function and evaluation metric are both set to Root Mean Squared Error (RMSE), the task_type is specified as CPU, the verbosity level is set to 100 (displaying progress every 100 iterations), and a random_seed of 42 is used for reproducibility. This configuration defines how the CatBoost model will be trained.

In [ ]:
# Train model directly with slightly adjusted parameters
final_model = CatBoostRegressor(
    iterations=2100,      # Slightly increased
    learning_rate=0.019,  # Small adjustment
    depth=7,             # Slightly decreased
    l2_leaf_reg=3.5,      # Small adjustment
    bagging_temperature=0.45, # Small adjustment
    random_strength=0.25, # Small adjustment
    loss_function="RMSE",
    eval_metric="RMSE",
    task_type="CPU",
    verbose=100,
    random_seed=42)

<font color = "DeepSkyBlue">**Train model**

This line of code initiates the training process for the final_model (which is a CatBoostRegressor object initialized in the previous step). The fit() method takes the training features (X) and the corresponding target variable (y) as input. During this process, the CatBoost model learns the relationship between the features and the target by iteratively building an ensemble of decision trees, aiming to minimize the specified loss function (RMSE in this case). The verbose=100 parameter in the model initialization ensures that training progress is printed every 100 iterations.

In [ ]:
# Train model
final_model.fit(X, y)

<font color = "DeepSkyBlue">**Predict test set**

This line uses the trained final_model (the CatBoostRegressor) to generate predictions on the unseen test dataset (X_test). The predict() method takes the features of the test set as input and outputs an array of predicted listening time values for each corresponding episode in the test set. These predictions are stored in the final_preds variable.

In [ ]:
# Predict test set
final_preds = final_model.predict(X_test)

<font color = "DeepSkyBlue">**Save submission file**

This code focuses on saving the final predictions to a submission file. It creates a pandas DataFrame named submission with two columns: "id" taken from the test DataFrame and "Listening_Time_minutes" which contains the final predictions (final_preds), clipped at a minimum value of 0 using np.clip. This clipping ensures that no negative listening times are included in the submission. The DataFrame is then saved as a CSV file named "catboost_submission.csv" in the /kaggle/working/ directory, without including the DataFrame index. Finally, a confirmation message "Submission saved." is printed to the console.

In [ ]:
# Save submission
submission = pd.DataFrame({
    "id": test["id"],
    "Listening_Time_minutes": np.clip(final_preds, 0, None)})
submission.to_csv("/kaggle/working/catboost_submission.csv", index=False)
print("Submission saved.")